## 🔧 Install & Import

> **Tối ưu RAM:** Import gọn, seed cố định, kiểm tra GPU/RAM ngay từ đầu.

In [ ]:
!pip install transformers datasets accelerate wordcloud -q
# ❌ Bỏ lightgbm — LightGBM convert sparse→dense tốn ~40GB RAM với 200k features

In [ ]:
import numpy as np
import pandas as pd
import re, os, gc, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
# ✅ Mixed precision: tăng tốc 2x, giảm VRAM ~40% trên P100
from torch.cuda.amp import autocast, GradScaler

import psutil

# --- Seed ---
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_vram:.1f} GB')

# --- Hàm theo dõi RAM (gọi sau mỗi bước lớn) ---
def ram_usage():
    p   = psutil.Process(os.getpid())
    ram = p.memory_info().rss / 1e9
    print(f'  💾 RAM đang dùng: {ram:.2f} GB')
    if DEVICE.type == 'cuda':
        vram = torch.cuda.memory_allocated() / 1e9
        print(f'  🎮 VRAM allocated: {vram:.2f} GB')

ram_usage()

---
## 📂 Load Data

In [ ]:
DATA_DIR = '/kaggle/input/datasets/namnguynnnn/sentence-classification/Dataset/'

train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')

print(f'Train: {train.shape} | Test: {test.shape}')

# Xử lý missing & duplicate
train['Text'] = train['Text'].fillna('')
test['Text']  = test['Text'].fillna('')

n_before = len(train)
train = train.drop_duplicates(subset=['Text']).reset_index(drop=True)
print(f'Xóa {n_before - len(train)} dòng duplicate → còn {len(train)} mẫu')

display(train.head(3))
ram_usage()

---
## 📊 EDA

> **Tối ưu RAM:** Tạo cột phụ cho EDA rồi **xóa ngay** sau khi dùng xong. Không giữ lại trong DataFrame.

In [ ]:
# --- 3.1 Label Distribution ---
label_counts = train['Label'].value_counts()
label_pct    = train['Label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x='Label', data=train, palette='Set2', ax=axes[0])
axes[0].set_title('Label Distribution')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')
label_pct.plot.pie(autopct='%1.1f%%', ax=axes[1],
                   labels=['Positive','Negative'],
                   colors=['#66c2a5','#fc8d62'], startangle=90)
axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

ratio = label_counts.max() / label_counts.min()
print(f'Imbalance ratio: {ratio:.2f}')
if ratio < 1.5:   print('✅ Balanced')
elif ratio < 3:   print('⚠️  Hơi lệch → dùng class_weight="balanced"')
else:             print('❌ Rất lệch → cần xử lý thêm')

In [ ]:
# --- 3.2 Text Length (tạo series tạm, không lưu vào train) ---
# ✅ Dùng .map() thay .apply() + không gán vào train để tiết kiệm RAM

char_len   = train['Text'].map(len)
word_count = train['Text'].map(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x=char_len,   hue=train['Label'], bins=50, ax=axes[0]); axes[0].set_title('Char Length')
sns.histplot(x=word_count, hue=train['Label'], bins=50, ax=axes[1]); axes[1].set_title('Word Count')
plt.tight_layout(); plt.show()

for pct in [50, 90, 95, 99]:
    print(f'{pct:>3}th pct word count: {np.percentile(word_count, pct):.0f}')

# ✅ Gợi ý MAX_LEN cho BERT dựa trên phân phối thực tế
p95 = int(np.percentile(word_count, 95))
RECOMMENDED_MAX_LEN = min(p95 * 2, 256)  # *2 vì tokenizer tạo ~2 token/word
print(f'\n→ Gợi ý MAX_LEN = {RECOMMENDED_MAX_LEN}')

# ✅ Xóa series tạm ngay sau khi dùng
del char_len, word_count
gc.collect()

In [ ]:
# --- 3.3 WordCloud ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, name, cmap, ax in [(1,'Positive','Greens',axes[0]),(0,'Negative','Reds',axes[1])]:
    text = ' '.join(train[train['Label'] == label]['Text'])
    wc = WordCloud(width=700, height=350, background_color='white',
                   colormap=cmap, max_words=80, random_state=SEED).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(f'WordCloud — {name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
del text, wc; gc.collect()

In [ ]:
# --- 3.4 Top Bigrams ---
def get_top_ngrams(texts, n=2, top_k=12):
    vec = CountVectorizer(ngram_range=(n,n), max_features=5000,
                          stop_words='english').fit(texts)
    bag = vec.transform(texts)
    freq = [(w, bag.sum(axis=0)[0, i]) for w, i in vec.vocabulary_.items()]
    return sorted(freq, key=lambda x: x[1], reverse=True)[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for label, name, color, ax in [(1,'Positive','#66c2a5',axes[0]),(0,'Negative','#fc8d62',axes[1])]:
    top = get_top_ngrams(train[train['Label']==label]['Text'], n=2, top_k=12)
    df_ng = pd.DataFrame(top, columns=['bigram','count'])
    sns.barplot(data=df_ng, x='count', y='bigram', color=color, ax=ax)
    ax.set_title(f'Top Bigrams — {name}')
plt.tight_layout(); plt.show()

---
## 🧹 Text Preprocessing

In [ ]:
STOPWORDS  = set(stopwords.words('english'))
KEEP_WORDS = {'not','no','never','neither','nor','hardly','barely'}
STOPWORDS -= KEEP_WORDS

CONTRACTIONS = {
    "don't":"do not","doesn't":"does not","didn't":"did not",
    "won't":"will not","wouldn't":"would not","can't":"cannot",
    "couldn't":"could not","shouldn't":"should not",
    "isn't":"is not","aren't":"are not","wasn't":"was not",
    "weren't":"were not","hadn't":"had not","hasn't":"has not",
    "haven't":"have not","mustn't":"must not","needn't":"need not",
    "it's":"it is","he's":"he is","she's":"she is",
    "i've":"i have","i'm":"i am","i'll":"i will","i'd":"i would",
}

def clean_text(text: str) -> str:
    """Clean text cho TF-IDF. KHÔNG dùng cho BERT."""
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)    # bỏ URL
    text = re.sub(r'<.*?>', '', text)              # bỏ HTML
    for c, f in CONTRACTIONS.items():             # expand contractions
        text = text.replace(c, f)
    text = re.sub(r'[^a-z\s]', ' ', text)         # chỉ giữ chữ cái
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_text'] = train['Text'].apply(clean_text)
test['clean_text']  = test['Text'].apply(clean_text)

print('Ví dụ clean:')
for i in range(2):
    print(f'\n[{i}] BEFORE: {train["Text"].iloc[i][:80]}')
    print(f'[{i}] AFTER:  {train["clean_text"].iloc[i][:80]}')
ram_usage()

---
## ⚡ Baseline: TF-IDF + ML

> **Tối ưu RAM:**
> - `dtype=np.float32` → giảm 50% RAM so với float64 mặc định
> - `max_features=30k` word + `20k` char → đủ cho medical domain, không OOM
> - Xóa matrix trung gian ngay sau `hstack`
> - ❌ Bỏ LightGBM — chuyển sparse sang dense nội bộ → OOM
> - ✅ Dùng `LinearSVC` thay thế — nhanh hơn, nhẹ hơn, thường tốt hơn LGBM trên TF-IDF

In [ ]:
# ============================================================
# 5.1 TF-IDF FEATURES — RAM-optimized
# ============================================================
print('Tạo TF-IDF features...')
ram_usage()

# ✅ dtype=float32: giảm 50% RAM (float64 là default)
# ✅ max_features=30k: đủ tốt cho medical review domain
# ✅ ngram (1,2) thay (1,3): bỏ trigram ít thông tin nhất
vec_word = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30_000,
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
    dtype=np.float32      # ← QUAN TRỌNG: giảm 50% RAM
)

# ✅ char (3,4): bỏ 2-gram (noise) và 5-gram (quá hiếm)
vec_char = TfidfVectorizer(
    ngram_range=(3, 4),
    max_features=20_000,
    sublinear_tf=True,
    min_df=2,
    analyzer='char_wb',
    dtype=np.float32      # ← QUAN TRỌNG
)

# Fit + transform
X_train_w = vec_word.fit_transform(train['clean_text'])
X_test_w  = vec_word.transform(test['clean_text'])
X_train_c = vec_char.fit_transform(train['clean_text'])
X_test_c  = vec_char.transform(test['clean_text'])

# Ghép
X_train_tfidf = sp.hstack([X_train_w, X_train_c], format='csr')
X_test_tfidf  = sp.hstack([X_test_w,  X_test_c],  format='csr')

# ✅ Xóa matrix trung gian NGAY — không cần giữ lại
del X_train_w, X_train_c, X_test_w, X_test_c
gc.collect()

y_train = train['Label'].values

print(f'Train TF-IDF: {X_train_tfidf.shape}')
print(f'Test  TF-IDF: {X_test_tfidf.shape}')
ram_usage()

In [ ]:
# ============================================================
# 5.2 CROSS-VALIDATION — chỉ dùng model nhẹ với sparse matrix
# ============================================================
# ❌ LightGBM: convert sparse→dense nội bộ → OOM với 50k features
# ✅ LogisticRegression (saga): hoạt động tốt với sparse, RAM thấp
# ✅ LinearSVC: thường tốt hơn LR trên text, vẫn sparse-compatible
#    (bọc CalibratedClassifierCV để có predict_proba cho ensemble)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

models = {
    'Logistic Regression': LogisticRegression(
        C=3, max_iter=500, solver='saga', class_weight='balanced'
    ),
    'LinearSVC': CalibratedClassifierCV(
        LinearSVC(C=0.5, max_iter=2000, class_weight='balanced'),
        cv=3
    )
}

print('Chạy Cross-Validation...\n')
results = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X_train_tfidf, y_train,
        cv=skf, scoring='f1_macro', n_jobs=1
    )
    results[name] = scores
    print(f'{name}:')
    print(f'  Fold scores: {[f"{s:.4f}" for s in scores]}')
    print(f'  Mean F1:     {scores.mean():.4f} ± {scores.std():.4f}\n')

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([results[m] for m in models], labels=list(models.keys()))
ax.set_title('F1-macro CV Scores'); ax.set_ylabel('F1-macro')
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 5.3 TRAIN BEST MODEL TRÊN TOÀN BỘ TRAIN SET
# ============================================================
best_model_name = max(results, key=lambda k: results[k].mean())
best_ml_model   = models[best_model_name]

print(f'Best model: {best_model_name} | CV F1: {results[best_model_name].mean():.4f}')
best_ml_model.fit(X_train_tfidf, y_train)
baseline_probs = best_ml_model.predict_proba(X_test_tfidf)  # (N_test, 2)
print('✅ Baseline predictions xong')
ram_usage()

---
## 🧠 BERT Fine-tuning + 3-Fold CV

> **Tối ưu RAM/VRAM cho P100 (16GB VRAM, ~13GB RAM):**
> - ✅ `N_FOLDS=3` thay 5 — giảm 40% thời gian, RAM tương đương
> - ✅ `BATCH_SIZE=32` + Mixed Precision (fp16) — P100 hỗ trợ tốt
> - ✅ `MAX_LEN=128` — phần lớn review bác sĩ < 128 token
> - ✅ `num_workers=0` — Kaggle P100 ít CPU core, num_workers>0 gây overhead
> - ✅ `pin_memory=True` — tăng tốc transfer CPU→GPU
> - ✅ Xóa EDA columns trước khi train BERT
> - ✅ `torch.cuda.empty_cache()` sau mỗi fold

In [ ]:
# ✅ Xóa cột EDA không cần thiết trước khi train BERT
# clean_text đã dùng cho TF-IDF, BERT dùng Text gốc
cols_to_drop = [c for c in ['clean_text'] if c in train.columns]
train.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=[c for c in cols_to_drop if c in test.columns], inplace=True)
gc.collect()
print(f'Columns còn lại: {train.columns.tolist()}')
ram_usage()

In [ ]:
# ============================================================
# CONFIG — Calibrated cho P100 Kaggle
# ============================================================
MODEL_NAME   = 'distilbert-base-uncased'

# ✅ MAX_LEN=128: đủ cho ~95% review bác sĩ, VRAM giảm 2x so với 256
# Nếu p95 word count > 70 từ → tăng lên 192 hoặc 256
MAX_LEN      = 128

# ✅ BATCH_SIZE=32 với fp16: P100 16GB VRAM chạy được
# Nếu OOM → giảm xuống 16
BATCH_SIZE   = 32

EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1

# ✅ 3 folds thay 5: tiết kiệm 40% thời gian, kết quả vẫn đáng tin cậy
N_FOLDS      = 3

# ✅ Mixed precision: tăng tốc 1.5-2x, giảm VRAM ~40%
USE_AMP      = True  # P100 hỗ trợ float16

print(f'Model:      {MODEL_NAME}')
print(f'MAX_LEN:    {MAX_LEN}')
print(f'BATCH_SIZE: {BATCH_SIZE}')
print(f'EPOCHS:     {EPOCHS}')
print(f'N_FOLDS:    {N_FOLDS}')
print(f'USE_AMP:    {USE_AMP}  (Mixed Precision fp16)')
print(f'Device:     {DEVICE}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'\n✅ Tokenizer loaded: {MODEL_NAME}')

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================
class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = list(texts)
        self.labels = list(labels) if labels is not None else None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print('✅ ReviewDataset ready')

In [ ]:
# ============================================================
# TRAINING FUNCTIONS — với Mixed Precision (AMP)
# ============================================================
scaler = GradScaler(enabled=USE_AMP)  # ✅ GradScaler cho fp16

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    n_batches  = len(loader)

    for i, batch in enumerate(loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # ✅ autocast: tự động chọn fp16/fp32 cho từng operation
        with autocast(enabled=USE_AMP):
            loss = model(**batch).loss

        # ✅ scaler: scale loss để tránh underflow với fp16
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)  # ✅ set_to_none: tiết kiệm VRAM hơn zero_()

        total_loss += loss.item()
        if (i + 1) % max(1, n_batches // 4) == 0:
            print(f'    [{i+1}/{n_batches}] loss={loss.item():.4f}')

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_probs = [], []
    for batch in loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
        with autocast(enabled=USE_AMP):
            logits = model(**inputs).logits
        probs = torch.softmax(logits.float(), dim=-1)  # float32 để đảm bảo chính xác
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

print('✅ Training functions ready')

In [ ]:
# ============================================================
# 3-FOLD CV + OOF
# ============================================================
texts_all  = train['Text'].values
labels_all = train['Label'].values
texts_test = test['Text'].values

oof_preds       = np.zeros(len(train), dtype=int)
oof_probs       = np.zeros((len(train), 2))
test_probs_list = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts_all, labels_all)):
    print(f'\n{"="*55}')
    print(f'FOLD {fold+1}/{N_FOLDS}  |  train={len(tr_idx)}  val={len(val_idx)}')
    print(f'{"="*55}')
    ram_usage()

    # --- DataLoader ---
    # ✅ num_workers=0: tránh overhead trên Kaggle (ít CPU core)
    # ✅ pin_memory=True: tăng tốc CPU→GPU transfer
    tr_loader   = DataLoader(ReviewDataset(texts_all[tr_idx],  labels_all[tr_idx]),
                             batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=0, pin_memory=True)
    val_loader  = DataLoader(ReviewDataset(texts_all[val_idx], labels_all[val_idx]),
                             batch_size=BATCH_SIZE*2, shuffle=False,
                             num_workers=0, pin_memory=True)
    test_loader = DataLoader(ReviewDataset(texts_test),
                             batch_size=BATCH_SIZE*2, shuffle=False,
                             num_workers=0, pin_memory=True)

    # --- Model ---
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(tr_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --- Training Loop ---
    best_f1, best_state = 0, None

    for epoch in range(EPOCHS):
        print(f'\n  Epoch {epoch+1}/{EPOCHS}')
        avg_loss = train_one_epoch(model, tr_loader, optimizer, scheduler)

        val_preds, _ = evaluate(model, val_loader)
        f1 = f1_score(labels_all[val_idx], val_preds, average='macro')
        print(f'  → loss={avg_loss:.4f}  val_F1={f1:.4f}')

        if f1 > best_f1:
            best_f1 = f1
            # ✅ Lưu trên CPU: không chiếm VRAM trong lúc train
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✅ New best: {best_f1:.4f}')

    # --- Predict với best model ---
    model.load_state_dict(best_state)
    model.to(DEVICE)
    oof_preds[val_idx], oof_probs[val_idx] = evaluate(model, val_loader)
    _, test_probs = evaluate(model, test_loader)
    test_probs_list.append(test_probs)

    print(f'\n  Fold {fold+1} Final F1: {f1_score(labels_all[val_idx], oof_preds[val_idx], average="macro"):.4f}')

    # ✅ Giải phóng VRAM ngay sau mỗi fold
    del model, optimizer, scheduler, best_state
    del tr_loader, val_loader, test_loader
    torch.cuda.empty_cache()
    gc.collect()
    ram_usage()

oof_f1 = f1_score(labels_all, oof_preds, average='macro')
print(f'\n{"="*55}')
print(f'🎯 OOF F1-macro (BERT {N_FOLDS}-fold): {oof_f1:.4f}')
print(f'{"="*55}')

---
## 📈 Evaluation & Error Analysis

In [ ]:
# --- Classification Report ---
print('=== BERT OOF Evaluation ===')
print(classification_report(labels_all, oof_preds,
      target_names=['Negative (0)', 'Positive (1)']))

# --- Confusion Matrix ---
cm = confusion_matrix(labels_all, oof_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg','Pred Pos'],
            yticklabels=['True Neg','True Pos'], ax=ax)
ax.set_title(f'OOF Confusion Matrix  |  F1={oof_f1:.4f}')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Neg:  {tn}  |  False Pos: {fp}')
print(f'False Neg: {fn}  |  True Pos:  {tp}')

In [ ]:
# --- Error Analysis ---
# ✅ Không dùng train.copy() — tạo DataFrame mới chỉ với cột cần thiết
error_df = pd.DataFrame({
    'Text':        train['Text'],
    'Label':       labels_all,
    'oof_pred':    oof_preds,
    'prob_pos':    oof_probs[:, 1],
    'confidence':  oof_probs.max(axis=1),
})
error_df['is_wrong'] = error_df['Label'] != error_df['oof_pred']

errors = error_df[error_df['is_wrong']].sort_values('confidence', ascending=False)
print(f'Tổng lỗi: {len(errors)} / {len(error_df)} ({len(errors)/len(error_df)*100:.1f}%)')
print(f'False Pos: {(errors["oof_pred"]==1).sum()}  |  False Neg: {(errors["oof_pred"]==0).sum()}')

print('\n--- 5 Lỗi tự tin nhất ---')
for _, r in errors.head(5).iterrows():
    print(f'\nText: {r["Text"][:100]}...')
    print(f'True={r["Label"]} | Pred={r["oof_pred"]} | Conf={r["confidence"]:.3f}')

del error_df; gc.collect()

---
## 📤 Inference & Submission

In [ ]:
# ============================================================
# ENSEMBLE: Average probability của N folds
# ============================================================
test_avg_probs   = np.stack(test_probs_list, axis=-1).mean(axis=-1)  # (N_test, 2)
bert_final_preds = test_avg_probs.argmax(axis=1)

print(f'BERT {N_FOLDS}-fold ensemble:')
print(f'  Negative: {(bert_final_preds==0).sum()}')
print(f'  Positive: {(bert_final_preds==1).sum()}')

In [ ]:
# ============================================================
# ENSEMBLE BERT + BASELINE
# ============================================================
# BERT hiểu ngữ nghĩa sâu, Baseline bắt được pattern từ vựng khác
# → Kết hợp thường tốt hơn từng model riêng lẻ

BERT_WEIGHT = 0.80
ML_WEIGHT   = 0.20

ensemble_probs = BERT_WEIGHT * test_avg_probs + ML_WEIGHT * baseline_probs
ensemble_preds = ensemble_probs.argmax(axis=1)

agreement = (bert_final_preds == ensemble_preds).mean()
print(f'Ensemble BERT({BERT_WEIGHT}) + ML({ML_WEIGHT}):')
print(f'  Negative: {(ensemble_preds==0).sum()}')
print(f'  Positive: {(ensemble_preds==1).sum()}')
print(f'  Đồng thuận với BERT-only: {agreement*100:.1f}%')

In [ ]:
# ============================================================
# THRESHOLD TUNING — tối ưu trên OOF
# ============================================================
best_thr, best_thr_f1 = 0.5, 0.0
for thr in np.arange(0.3, 0.71, 0.01):
    preds = (oof_probs[:, 1] >= thr).astype(int)
    f1    = f1_score(labels_all, preds, average='macro')
    if f1 > best_thr_f1:
        best_thr_f1 = f1
        best_thr    = thr

print(f'Default threshold 0.5: F1 = {oof_f1:.4f}')
print(f'Optimal threshold {best_thr:.2f}: F1 = {best_thr_f1:.4f} (+{(best_thr_f1-oof_f1)*100:.2f}%)')

final_preds_tuned = (test_avg_probs[:, 1] >= best_thr).astype(int)

In [ ]:
# ============================================================
# TẠO SUBMISSION — chọn predictions tốt nhất
# ============================================================
# Thứ tự ưu tiên:
# 1. ensemble_preds + threshold tuning
# 2. bert_final_preds + threshold tuning
# 3. bert_final_preds (fallback)

final_predictions = (ensemble_probs[:, 1] >= best_thr).astype(int)

submission = pd.DataFrame({
    'ID':    range(1, len(test) + 1),
    'Label': final_predictions
})
submission.to_csv('submission.csv', index=False)

print('✅ submission.csv tạo xong!')
display(submission.head(5))
print(submission['Label'].value_counts())

In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================
sub = pd.read_csv('submission.csv')
checks = [
    ('Số dòng đúng',    len(sub) == len(test)),
    ('Có cột ID',       'ID' in sub.columns),
    ('Có cột Label',    'Label' in sub.columns),
    ('Label chỉ 0/1',   set(sub['Label'].unique()).issubset({0,1})),
    ('Không có NaN',    sub.isnull().sum().sum() == 0),
    ('ID từ 1 đến N',   sub['ID'].min()==1 and sub['ID'].max()==len(test)),
]
print('=== SANITY CHECK ===')
all_ok = True
for name, result in checks:
    print(f'{"✅" if result else "❌"} {name}')
    all_ok = all_ok and result
print()
print('🚀 Sẵn sàng submit!' if all_ok else '⚠️ Có lỗi!')

print(f'\n=== SUMMARY ===')
print(f'Baseline CV F1:  {results[best_model_name].mean():.4f}')
print(f'BERT OOF F1:     {oof_f1:.4f}')
print(f'Best threshold:  {best_thr:.2f}  →  F1={best_thr_f1:.4f}')